In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("medical_students_dataset.csv")


print("Dataset loaded successfully.")
df.head()

## 3.1 Understanding the Dataset

In [ ]:
# Number of records and attributes
rows, columns = df.shape

print("Number of records (rows):", rows)
print("Number of attributes (columns):", columns)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)


In [ ]:
# Statistical summary
# include='all' shows statistics for both numeric and categorical columns.
df.describe(include="all").T


In [ ]:
# Distribution of numeric variables
numeric_columns = df.select_dtypes(include=np.number).columns

df[numeric_columns].hist(
    figsize=(15, 12),
    bins=30
)
plt.suptitle("Distribution of Numeric Variables")
plt.tight_layout()
plt.show()


In [ ]:
# Count missing values in each column
missing_values = df.isnull().sum().sort_values(ascending=False)

print("Missing values by column:")
print(missing_values)

print("\nTotal missing cells:", int(df.isnull().sum().sum()))


In [ ]:
# Count duplicate rows
duplicate_count = df.duplicated().sum()
print("Number of duplicate rows:", int(duplicate_count))


## 3.2 Data Cleaning


In [ ]:
# Keep track of the dataset size before cleaning
before_cleaning = df.shape

# 1. Drop rows where the target is missing
df = df.dropna(subset=["Diabetes"]).copy()

# 2. Remove duplicate rows
df = df.drop_duplicates(keep="first").copy()

# 3. Fill numeric columns with their column-wise mean
numeric_columns = df.select_dtypes(include=np.number).columns
for col in numeric_columns:
    df[col] = df[col].fillna(df[col].mean())

# 4. Fill categorical columns with their mode
categorical_columns = df.select_dtypes(include="object").columns
for col in categorical_columns:
    mode_value = df[col].mode(dropna=True)[0]
    df[col] = df[col].fillna(mode_value)

print("Shape before cleaning:", before_cleaning)
print("Shape after cleaning:", df.shape)
print("\nRemaining missing values:")
print(df.isnull().sum())


## 3.3 Creation of Input and Output Features

In [ ]:
# Features = all columns except target
# Labels = target column only
Features = df.drop(columns=["Diabetes"]).copy()
Labels = df["Diabetes"].copy()

print("Features shape:", Features.shape)
print("Labels shape:", Labels.shape)

print("\nFeature columns:")
print(Features.columns.tolist())

print("\nTarget values:")
print(Labels.value_counts())


## 3.4 Conversion of Features into Numeric Values


In [ ]:
# Identify categorical feature columns
categorical_feature_columns = Features.select_dtypes(include="object").columns.tolist()
print("Categorical feature columns:", categorical_feature_columns)

# One-hot encode all categorical feature columns.
# drop_first=False keeps the complete one-hot representation.
Features_encoded = pd.get_dummies(
    Features,
    columns=categorical_feature_columns,
    dtype=int
)

# Convert target labels to numeric values
Labels_encoded = Labels.map({"No": 0, "Yes": 1}).astype(int)

print("\nEncoded Features shape:", Features_encoded.shape)
print("\nEncoded feature columns:")
print(Features_encoded.columns.tolist())

print("\nFirst 5 rows of encoded features:")
display(Features_encoded.head())


## 3.5 Scaling of the Features

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler


one_hot_columns = [
    col for col in Features_encoded.columns
    if any(col.startswith(original + "_") for original in categorical_feature_columns)
]

# The remaining columns are numeric/original continuous features.
scale_columns = [col for col in Features_encoded.columns if col not in one_hot_columns]

print("Columns to scale:")
print(scale_columns)

print("\nOne-hot columns not scaled:")
print(one_hot_columns)


In [ ]:
def scale_features(features, method="standard"):
    """Scale only original numeric features.

    method='standard' -> StandardScaler
    method='minmax'   -> MinMaxScaler
    """
    scaled = features.copy()

    if method.lower() == "standard":
        scaler = StandardScaler()
    elif method.lower() == "minmax":
        scaler = MinMaxScaler()
    else:
        raise ValueError("method must be 'standard' or 'minmax'")

    scaled[scale_columns] = scaler.fit_transform(scaled[scale_columns])
    return scaled, scaler


# Both required scaling methods
Features_standard, standard_scaler = scale_features(Features_encoded, "standard")
Features_minmax, minmax_scaler = scale_features(Features_encoded, "minmax")

print("Standard-scaled feature sample:")
display(Features_standard.head())

print("Min-Max-scaled feature sample:")
display(Features_minmax.head())


In [ ]:
# Verify the two scaling methods
comparison = pd.DataFrame({
    "Original Min": Features_encoded[scale_columns].min(),
    "Original Max": Features_encoded[scale_columns].max(),
    "Standard Mean": Features_standard[scale_columns].mean(),
    "Standard Std": Features_standard[scale_columns].std(),
    "MinMax Min": Features_minmax[scale_columns].min(),
    "MinMax Max": Features_minmax[scale_columns].max()
})

comparison.round(4)


In [ ]:

SCALING_METHOD = "standard"

if SCALING_METHOD == "standard":
    Features_processed = Features_standard.copy()
else:
    Features_processed = Features_minmax.copy()

print("Selected scaling method:", SCALING_METHOD)
print("Processed feature shape:", Features_processed.shape)


## 3.6 Correlation Analysis

In [ ]:
# Combine processed features with the numeric target
correlation_data = Features_processed.copy()
correlation_data["Diabetes"] = Labels_encoded.values

# Correlation of every processed feature with the target
target_correlation = (
    correlation_data.corr(numeric_only=True)["Diabetes"]
    .drop("Diabetes")
    .sort_values(key=lambda x: x.abs(), ascending=False)
)

print("Correlation of every feature with Diabetes:")
display(target_correlation.to_frame(name="Correlation with Diabetes"))


In [ ]:
# Correlation matrix
plt.figure(figsize=(14, 10))
plt.imshow(correlation_data.corr(numeric_only=True), aspect="auto")
plt.colorbar(label="Correlation")
plt.title("Correlation Matrix")
plt.xticks(
    range(len(correlation_data.columns)),
    correlation_data.columns,
    rotation=90
)
plt.yticks(
    range(len(correlation_data.columns)),
    correlation_data.columns
)
plt.tight_layout()
plt.show()


In [ ]:

top_10_features = target_correlation.abs().sort_values(ascending=False).head(10).index

top_10_correlation = target_correlation.loc[top_10_features].sort_values(
    key=lambda x: x.abs(),
    ascending=False
)

print("Top 10 features by absolute correlation with Diabetes:")
display(top_10_correlation.to_frame(name="Correlation with Diabetes"))


In [ ]:
# 1D scatter plots for the top 10 correlated features.
# A sample is used only for visualization so the notebook remains responsive.
plot_data = correlation_data[list(top_10_features) + ["Diabetes"]].copy()

sample_size = min(5000, len(plot_data))
plot_sample = plot_data.sample(sample_size, random_state=42)

for feature in top_10_features:
    plt.figure(figsize=(8, 4))

    # Small vertical jitter makes overlapping binary target classes easier to see.
    y = plot_sample["Diabetes"] + np.random.default_rng(42).normal(
        0, 0.025, size=len(plot_sample)
    )

    plt.scatter(plot_sample[feature], y, alpha=0.25, s=10)
    plt.xlabel(feature)
    plt.ylabel("Diabetes (0 = No, 1 = Yes)")
    plt.title(f"1D Scatter Plot: {feature} vs Diabetes")
    plt.yticks([0, 1], ["No", "Yes"])
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()
